# သင်ခန်းစာ ၁၃ - ကိုယ်စားလှယ်မှတ်ဉာဏ်


## Setup

ဤ notebook သည် **Microsoft Agent Framework** (MAF) ကို အသုံးပြု၍ **persistent memory** ပါသည့် ခရီးသွားဘွတ်ကင် အေးဂျင့်ကို မည်သို့ တည်ဆောက်ရမည်ကို ပြသသည်။

သင်သည် အေးဂျင့်မှတ်ဉာဏ်အမျိုးအစားများ — အလုပ်လုပ်မှု၊ short-term နှင့် long-term — များက မည်သို့ အေးဂျင့်သည် စကားပြောမှုများအတွင်း သတင်းအချက်အလက်များကို သိမ်းဆည်း အသုံးပြုခြင်းအား သက်ရောက်စေသည်ကို သင်ယူမည်ဖြစ်သည်။

**လိုအပ်ချက်များ:**
- Azure AI Foundry ပရောဂျက်တစ်ခု သာလျှင် chat မော်ဒယ် တင်ထားပြီး (ဥပမာ `gpt-4o-mini`) ဖြစ်ရမည်။
- Azure CLI ဖြင့် မှတ်ပုံတင်ထားပြီး — သင့် terminal တွင် `az login` ကို ရေးထုတ်ပါ။
- `AZURE_AI_PROJECT_ENDPOINT` — သင့် Azure AI Foundry ပရောဂျက် endpoint။
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — သင့်တင်ထားသော မော်ဒယ်အမည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## အီးဂျင့်မှတ်ဉာဏ်အမျိုးအစားများ

AI အီးဂျင့်များသည် အမျိုးမျိုးသောမှတ်ဉာဏ်များကို အသုံးပြုနိုင်ပြီး၊ တစ်ခုချင်းစီမှာ ထူးခြားသော ရည်ရွယ်ချက်ရှိသည်-

### အလုပ်လုပ်မှတ်ဉာဏ်
စကားပြောဆိုမှုတစ်ကြောင်း၏စာတိုက်ကြောင်းများ - တစ်ဦးတည်းသောအစည်းအဝေးတွင် လဲလှယ်သော စာတိုက်များ ဖြစ်သည်။ အီးဂျင့်သည် coherence ကို ထိန်းသိမ်းရန်တစ်ချက်ထဲတွင် ယခင်စာတိုက်များကို သွားလည်ကြည့်ရန် ရနိုင်သည်။ MAF တွင် ဤအရာကို **`agent.create_session()`** မှတဆင့် ဖန်တီးပြီး `AgentSession` ကို ပြန်လည်ပေးမည်ဖြစ်သည်။

### အချိန်တိုမှတ်ဉာဏ်
တာဝန် သို့မဟုတ်အစည်းအဝေးတစ်ခု၏ ကြာမြင့်ချိန်အတွင်း တည်ရှိနေသော်လည်း အမြဲတမ်း သိမ်းဆည်းမထားသော အချက်အလက်များဖြစ်သည်။ ဥပမာအားဖြင့် အီးဂျင့်သည် တစ်ခုထပ်တလဲလဲကြံဆောင်မှုဆွေးနွေးပွဲကာလအတွင်း အချက်အလက်များကို တွဲဆုံ စုဆောင်းကာ နောက်ဆုံး အစီအစဉ် တစ်ခုကို ထုတ်ပေးနိုင်သည်။

### ရေရှည်မှတ်ဉာဏ်
အစည်းအဝေးများ **အချင်းချင်း ကျော်လွန်၍** တည်ရှိနေသော နှစ်သက်ချက်များနှင့် အချက်အလက်များ ဖြစ်သည်။ ပြန်လည်လာသော အသုံးပြုသူသည် ကိုယ်စားအစားအစာကန့်သတ်ချက်များ သို့မဟုတ် ခရီးသွားစတိုင်ကို ထပ်မမိန့်ကြားရပါ။ ရေရှည်မှတ်ဉာဏ်သည် ပုံမှန်အားဖြင့် ပြင်ပသိုလှောင်နေရာ၊ ဒေတာဘေ့စ်၊ ဖိုင် သို့မဟုတ် ဗက်တာ စာရင်းကို ထောက်ပံ့ပြီး tools များမှတဆင့် အီးဂျင့်ထံသို့ မြင်သာစေသည်။


## ဆွေးနွေးပွဲများဖြင့် အလုပ်လုပ်နိုင်သော နေရာသတိရမှု

အရင်းရှည်ဆုံးသော သတိရမှု ပုံစံမှာ ဆွေးနွေးပွဲ session ဖြစ်သည်။ တူညီသော session ကို (`agent.create_session()` မှဖန်တီးထားပြီး) ဆက်တိုက် `agent.run()` ခေါ်ဆိုမှုများထဲ ကျရောက်သလို သင်ပေးလိုက်သောအခါ၊ agent သည် အဆိုပါ ဆွေးနွေးပွဲ၏ ပြီးခဲ့သည့်ဖြစ်ရပ်များအားလုံးကို မြင်နိုင်ပြီး ဦးတည်ခဲ့သည့်အသေးစိတ်များကို ပြန်လည်မှတ်ယူနိုင်သည်။

ဆိုင်ခန်းခရီးသွား agent တစ်ဦး ဖန်တီးပြီး အလုပ်လုပ်နိုင်သော memory ကို ပြသပေးကြပါစို့။


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

အဲဂျင့်သည် ဘတ်ဂျက်ကို တိကျမှုရှိစွာ ပြန်ဖြေဆိုနိုင်ခဲ့သည်မှာ နှစ်ခုသော ချန်နယ် မက်ဆေ့ဂျ์များသည် အတူတူ ဆက်သွယ်မှု အစည်းအဝေးသာရှိသောကြောင့်ဖြစ်သည်။ ၎င်းသည် **အလုပ်လုပ်နေသောမှတ်ဉာဏ်** ဖြစ်ပြီး — ဆက်သွယ်မှု၏ ရှင်သန်ချိန်အတွင်းမှာသာ တည်ရှိသည်။

### ချန်နယ်အသစ်ဖြင့် ထွက်ပေါ်လာသော အချက်

**အသစ်** အစည်းအဝေးတစ်ခု ဖန်တီးခဲ့ပါက၊ အဲဂျင့်အနေနှင့် ယခင် ဆွေးနွေးမှုကို မမွတ်မံနိုင်ပါ။


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## နိဒါန်းကြာရှည်မှတ်ဉာဏ်ပုံစံ

အသုံးပြုသူအကြိုက်များကို **အစည်းအဝေးအတွင်း** မှတ်မိရန်၊ ဆက်သွယ်မှုတစ်ခုလုံးအပြင်ရှိတည်‌ထောင်ထားသော စတိုးဆိုင်ရာတစ်ခု လိုအပ်သည်။ အဲဒီစတိုးဆိုင်ရာသို့ဝင်ခွင့်ရရန်ကိုယ်စားလှယ်သည် **ကိရိယာများ** မှတဆင့် ခေါ်ယူနိုင်သော ဖန်ဆင်းထားသော အလုပ်များဖြစ်သည်။

အောက်တွင် ရိုးရှင်းသည့် အတွင်းမှတ်ဉာဏ်အကြိုက်များသိမ်းဆည်းရာစနစ်တစ်ခုကို အကောင်အထည်ဖော်ပေးထားပြီး (ထုတ်ကုန်တွင် ဒေတာဘေ့စ် သို့မဟုတ် ဗက်တာ အညွှန်းဖြင့် ရှေ့ခံပေါ်မှာ ထောက်ပံ့ပါမည်) ကိုယ်စားလှယ်သည် အသုံးပြုနိုင်သော ကိရိယာများအဖြစ် ထုတ်လွှင့်ထားသည်။

### မူဝါဒ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — ပထမဦးဆုံးအသုံးပြုသူသည် နှစ်ပတ်လည်ခရီးစဥ်ကို စီစဉ်သည်

Sarah သည် ပထမဦးဆုံးလာရောက်ပါတယ်။ အေးဂျင့်သည် သူမ၏Preferences များကို ကိရိယာများမှတဆင့် သိမ်းဆည်းပြီး ဟိုတယ်များကို အကြံပြုရန် အသုံးပြုသင့်သည်။


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah returns weeks later

Sarah သည် **အမှတ်တရသစ် Thread** အသစ်တစ်ခုစတင်သည် (session အသစ်သုံးခြင်းကို အတုယူခြင်း)။ အလုပ်လုပ်မည့်မှတ်ဉာဏ်သည် ဖွင့်လှစ်မှုမရှိသေးပေမယ့် ရေရှည်ကြိုက်နှစ်သက်မှုအချက်အလက်များမှာ ထည့်သွင်းထားသည်။ အေးဂျင့်သည် ၎င်းကို ရယူပြီး တိုက်ရိုက်အကြံပြုချက်များကို ပုဂ္ဂိုလ်ရေးပြုလုပ်ရန်အသုံးပြုသင့်သည်။


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## အကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် သင်သည် သုံးမျိုးသော ကိုယ်စားလှယ်မှတ်ဉာဏ်အမျိုးအစားများနှင့် အဲဒီအတိုင်း Microsoft Agent Framework ဖြင့် ပြုလုပ်ပုံကို သင်ယူခဲ့သည်-

| မှတ်ဉာဏ်အမျိုးအစား | MAF လုပ်ဆောင်ချက် | အသက်တမ်း |
|---|---|---|
| **အလုပ်လုပ်နေသော** | `agent.create_session()` | တစ်ခုတည်းစကားပြောမှု |
| **အနည်းငယ်ကာလ** | အကြောင်းအရာစုစည်းချက် တစ်ခုခုအတွင်းက | တစ်ခုတည်း ဂုဏ်ယူမှု / အစည်းအဝေး |
| **ရှည်သော ကာလ** | `@tool` လုပ်ဆောင်မှုများဖြင့် ဝင်ရောက်သုံးစွဲသော ပြင်ပသိုလှောင်မှု | အစည်းအဝေးအတွင်းကျော်လွှား၍ |

### အဓိကယူဆောင်ချက်များ
1. **`agent.create_session()`** သည် အလုပ်လုပ်နေသော မှတ်ဉာဏ်ကို ပေးသည် — ကိုယ်စားလှယ်သည် အစည်းအဝေးအတွင်းရှိ စကားပြောမှုမှတ်တမ်းအပြည့်အစုံကိုမြင်နိုင်သည်။
2. **အသစ်အစည်းအဝေးများသည် အကြောင်းအရာကို တတ်မနေကြပါ** — ရှည်လျားသော မှတ်ဉာဏ်မရှိလျှင် ကိုယ်စားလှယ်သည် ယခင်စကားပြောမှုများကို ပြန်မဖတ်နိုင်ပါ။
3. **`@tool` လုပ်ဆောင်ချက်များသည် ချိတ်ဆက်ပေးသည်** — ၎င်းတို့သည် ကိုယ်စားလှယ်အား အမြဲတမ်းသိုလှောင်မှုမှ အချက်အလက်များ သိမ်းဆည်းရန်နှင့် ပြန်ယူရန်ခွင့်ပြုသည်။
4. **အမူအရာများ တိုးတက်ကောင်းမွန်လာသည်** — စုဆောင်းထားသည့် နှစ်သက်မှုများအသေးစိတ်များပိုမိုရှိလာသည့်အတွက် ကိုယ်စားလှယ်၏ အကြံပြုချက်များ ပိုမိုကောင်းမွန်လာသည်။

### အမှန်တကယ် အသုံးချနိုင်မှုများ
- **ဖောက်သည်ဝန်ဆောင်မှု**: ဖောက်သည်မှတ်တမ်းများနှင့် နှစ်သက်ချက်များသိမ်းဆည်းခြင်း
- **ကိုယ်ပိုင် အကူအညီပေးသူများ**: ရက်ပေါင်းများသို့မဟုတ် အပတ်အထိ အကြောင်းအရာ ထိန်းသိမ်းခြင်း
- **ကျန်းမာရေး**: လူနာအချက်အလက်များနှင့် နှစ်သက်ချက်များ လိုက်လံစစ်ဆေးခြင်း
- **အီလက်ထရွန်နစ် စျေးဝယ်မှု**: မှတ်တမ်းအခြေခံသည့် ကိုယ်ပိုင်စျေးဝယ်ခြင်း

### နောက်တစ်ဆင့်များ
- မှတ်ဉာဏ်အတွင်းက dict ကို ဒေတာဘေ့စ် သို့မဟုတ် ဗက်တာစတိုး (ဥပမာ- Azure AI Search) ဖြင့် အစားထိုးခြင်း
- အချိန်ဟန်ချက်ကို တွက်ချက်ရန် မှတ်ဉာဏ် သက်တမ်းကုန်ဆုံးမှု ထည့်သွင်းခြင်း
- များပြားသော ကိုယ်စားလှယ်စနစ်များကို မျှဝေသည့် မှတ်ဉာဏ်ဖြင့် ဆောက်လုပ်ခြင်း
- knowledge-graph-backed memory အတွက် Cognee notebook ကို စူးစမ်းလေ့လာခြင်း


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**အာမခံချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ မှန်ကန်မှုကို ကြိုးစားထားသော်လည်း အလိုအလြောဖြစ်သော ဘာသာပြန်ချက်များတွင် အမှားအယွင်းများ ပါဝင်နိုင်ကြောင်း သိရှိရန်လိုအပ်ပါသည်။ မူရင်းစာတမ်းကို မူလဘာသာဖြင့်သာ အတည်ပြုရင်းမြစ်အဖြစ် အယူခံသင့်ပါသည်။ အရေးကြီးသော အချက်အလက်များအတွက် ကုသမှုအနေနှင့် လူ့ဘာသာပြန်သူမှ ပြုလုပ်သော ဘာသာပြန်မှုကို တိုက်တွန်းပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုမှုပေါ်တွင် ဖြစ်ပေါ်လာတတ်သော နားလည်မှုမှားယွင်းခြင်းများ၊ အဓိပ္ပါယ် ပျက်ကွက်မှုများအတွက် ကျွန်ုပ်တို့မှာ တာဝန်ယူမှု မရှိပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
